In [4]:
%pip install opencv-python
%pip install rasterio

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------- ----------------- 22.8/40.2 MB 111.1 MB/s eta 0:00:01
   ---------------------------------------  40.1/40.2 MB 134.2 MB/s eta 0:00:01
   ---------------------------------------  40.1/40.2 MB 134.2 MB/s eta 0:00:01
   ---------------------------------------  40.1/40.2 MB 134.2 MB/s eta 0:00:01
   ---------------------------------------- 40.2/40.2 MB 41.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------------- 12.3/12.3 MB 110.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/30.1 MB ? eta -:--:--
   ------------------------------ --------- 23.3/30.1 MB 113.1 MB/s eta 0:00:01
   ---------------------------------------  29.9/30.1 MB 90.0 MB/s eta 0:00:01
   ---------------------------------------- 30.1/30.1 MB 61.6 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.2.1
    Uninstalling click-8.2.1:
      Successfully uninstalled click-8.2.1
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Data Preparation Script for Sentinel-1 SAR Imagery

This script automates the extraction and preprocessing of Sentinel-1 SAR data for use in Super-Resolution (SR) / ESRGAN training. It addresses common channel mismatch errors encountered when feeding SAR images into models pretrained on RGB data (like VGG-19).

### What Happens When You Run It

1. **Reads the ZIP Archive**  
   The script locates and reads your 1.7 GB `.zip` file from the `data/` folder.

2. **Extracts the .SAFE Structure**  
   It unpacks the archive, preserving the standard Sentinel-1 `.SAFE` directory structure which contains measurement files and metadata.

3. **Navigates to Measurement Files**  
   Inside `.SAFE/measurement/`, it finds the two polarization TIFF files:  
   - **VH** (Vertical transmit, Horizontal receive) – ignored.  
   - **VV** (Vertical transmit, Vertical receive) – selected for processing.

4. **Processes the VV Image**  
   The script reads the VV-polarized TIFF, normalizes and tiles it into smaller patches suitable for training.

5. **Creates HR/LR Image Pairs**  
   It generates:
   - **High-Resolution (HR)** patches (full detail) saved to `train_HR/`.
   - **Low-Resolution (LR)** patches (downsampled versions) saved to `train_LR/`.  
   Both are saved as 3-channel PNG files (by replicating the single channel three times) to mimic RGB format.

6. **Bypasses VGG‑19 Channel Error**  
   By converting the single-channel SAR data to a 3-channel PNG, the images become compatible with pre-trained RGB networks like VGG‑19, which expect 3 input channels. This avoids the common "channel mismatch" error in ESRGAN/SRGAN repositories.

### Result

You get a curated dataset of HR/LR image pairs, ready to be fed into any super-resolution model without further modification.

In [10]:
import os
import zipfile
import glob
import cv2
import numpy as np
import rasterio

# --- Configuration ---
# Point this to your downloaded zip file in your data folder
ZIP_PATH = "../data/S1A_IW_GRDH_1SDV_20230115T052548_20230115T052613_046790_059C18_9C58.SAFE.zip" 
EXTRACT_DIR = "../data/" # Where the .SAFE folder will be unzipped

# Output directories for the GAN
OUTPUT_DIR = "../data/SAR_Dataset"
HR_DIR = os.path.join(OUTPUT_DIR, "train_HR")
LR_DIR = os.path.join(OUTPUT_DIR, "train_LR")

PATCH_SIZE_HR = 256
SCALE_FACTOR = 4
PATCH_SIZE_LR = PATCH_SIZE_HR // SCALE_FACTOR

# Create necessary directories
os.makedirs(HR_DIR, exist_ok=True)
os.makedirs(LR_DIR, exist_ok=True)
os.makedirs(EXTRACT_DIR, exist_ok=True)

def extract_and_find_tiff(zip_path, extract_to):
    print(f"Extracting {zip_path} to {extract_to}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    
    # The extracted folder has the same name as the zip (without the .zip extension)
    safe_dir_name = os.path.basename(zip_path).replace('.zip', '')
    safe_dir_path = os.path.join(extract_to, safe_dir_name)
    
    # Dynamically search for the VV polarization TIFF in the measurement folder
    search_pattern = os.path.join(safe_dir_path, "measurement", "*vv*.tiff")
    tiff_files = glob.glob(search_pattern)
    
    if not tiff_files:
        raise FileNotFoundError(f"Could not find a VV TIFF matching {search_pattern}")
        
    target_tiff = tiff_files[0]
    print(f"Successfully located target TIFF: {target_tiff}")
    return target_tiff

def process_sar_to_dataset(tiff_path):
    print(f"Loading massive SAR image into memory...")
    with rasterio.open(tiff_path) as src:
        # Read the single VV band
        image = src.read(1).astype(np.float32)
    
    h, w = image.shape
    print(f"Original SAR Image Shape: {h}x{w}")

    # 1. Normalize and Clip Outliers
    print("Normalizing radar backscatter anomalies...")
    p_low, p_high = np.percentile(image, (1, 99))
    image = np.clip(image, p_low, p_high)
    
    # Scale to 0-255 (8-bit)
    image_8bit = ((image - p_low) / (p_high - p_low + 1e-8) * 255.0).astype(np.uint8)

    # 2. Convert 1-channel to 3-channel (Fake RGB for VGG Perceptual Loss)
    image_rgb = cv2.cvtColor(image_8bit, cv2.COLOR_GRAY2RGB)

    # 3. Slice into patches
    print("Slicing into HR and LR patches...")
    patch_count = 0
    stride = PATCH_SIZE_HR 
    
    for y in range(0, h - PATCH_SIZE_HR, stride):
        for x in range(0, w - PATCH_SIZE_HR, stride):
            hr_patch = image_rgb[y:y + PATCH_SIZE_HR, x:x + PATCH_SIZE_HR]
            
            # Skip empty or mostly black patches (edges of the radar swath)
            if np.mean(hr_patch) < 10: 
                continue

            # Create LR patch via blur and bicubic downsampling
            blurred = cv2.GaussianBlur(hr_patch, (3, 3), 0)
            lr_patch = cv2.resize(blurred, (PATCH_SIZE_LR, PATCH_SIZE_LR), interpolation=cv2.INTER_CUBIC)

            # Save pairs to the respective folders
            hr_filename = os.path.join(HR_DIR, f"patch_{patch_count:06d}.png")
            lr_filename = os.path.join(LR_DIR, f"patch_{patch_count:06d}.png")
            
            cv2.imwrite(hr_filename, hr_patch)
            cv2.imwrite(lr_filename, lr_patch)
            
            patch_count += 1
            if patch_count % 500 == 0:
                print(f"Generated {patch_count} pairs...")

    print(f"Dataset creation complete! Total valid pairs generated: {patch_count}")

# --- Run the pipeline ---
if __name__ == "__main__":
    # Step 1: Unzip and locate the file
    target_tiff_path = extract_and_find_tiff(ZIP_PATH, EXTRACT_DIR)
    
    # Step 2: Process the TIFF into GAN patches
    process_sar_to_dataset(target_tiff_path)

Extracting ../data/S1A_IW_GRDH_1SDV_20230115T052548_20230115T052613_046790_059C18_9C58.SAFE.zip to ../data/...
Successfully located target TIFF: ../data/S1A_IW_GRDH_1SDV_20230115T052548_20230115T052613_046790_059C18_9C58.SAFE\measurement\s1a-iw-grd-vv-20230115t052548-20230115t052613-046790-059c18-001.tiff
Loading massive SAR image into memory...
Original SAR Image Shape: 16668x26549
Normalizing radar backscatter anomalies...
Slicing into HR and LR patches...
Generated 500 pairs...
Generated 1000 pairs...
Generated 1500 pairs...
Generated 2000 pairs...
Generated 2500 pairs...
Generated 3000 pairs...
Generated 3500 pairs...
Generated 4000 pairs...
Generated 4500 pairs...
Generated 5000 pairs...
Generated 5500 pairs...
Generated 6000 pairs...
Dataset creation complete! Total valid pairs generated: 6435
